# 实验 10　构建可追溯问答助手

## 任务要求

1. 设计至少 4 类可回答的问题，例如“谁出场最多”“哪几章曹操最活跃”“谁与诸葛亮共现最多”“哪些章节最像战争篇”。
2. 回答问题时，不只给结论，还要给出对应的依据来源（统计表、图或中间文件）。
3. 允许用大模型润色回答，但最终答案必须由数据结果驱动，而不是凭空生成。
4. 用一段总结说明：AI 帮了什么，又有哪些地方必须由自己判断。

## 可追溯的含义

可追溯回答 = **结论 + 可验证的证据文件**。每个答案都只读取前面实验已经落盘的 `outputs/tables/*.csv`，用户可以自己打开表格复核，从而避免大模型编造统计值，也便于调试和迭代。

> 运行顺序提示：请先执行实验 5—9，确保 `outputs/tables/` 中已经存在人物频次、逐章出场、共现边表、关键词、相似度与主题表。

In [1]:
# 让 notebook 在 notebooks/ 或仓库根目录下都能找到 src/ 包
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "sanguo_nlp").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from sanguo_nlp.paths import FIGURES_DIR, TABLES_DIR, ensure_output_dirs

ensure_output_dirs()
print("已定位仓库根目录:", ROOT.name)

已定位仓库根目录: sanguo-nlp-lab


In [2]:
import pandas as pd
from IPython.display import display

from sanguo_nlp.qa import SanguoQA

qa = SanguoQA()
print("已加载的中间结果：")
for name in ["character_frequencies", "character_trends", "edges", "keywords", "similarity", "topics"]:
    print(f"  {name}: {getattr(qa, name).shape}")

已加载的中间结果：
  character_frequencies: (16, 2)
  character_trends: (120, 17)
  edges: (118, 3)
  keywords: (1200, 3)
  similarity: (7140, 3)
  topics: (120, 10)


## 问题 1：谁出场最多？

In [3]:
answers = []

result, evidence = qa.who_appears_most(5)
display(result)
print("依据：", evidence)
answers.append(("谁出场最多？", result, evidence))

,人物,频次
0,刘备,2224
1,诸葛亮,1901
2,曹操,1037
3,关羽,981
4,赵云,447


依据： 依据 outputs/tables/character_frequencies.csv（别名归一后全书提及次数）；可用 outputs/tables/character_trends.csv 的逐章求和交叉验证：刘备=2211、诸葛亮=1880、曹操=1013、关羽=969、赵云=441


## 问题 2：哪几章曹操最活跃？

In [4]:
result, evidence = qa.most_active_chapters("曹操", 5)
display(result)
print("依据：", evidence)
answers.append(("哪几章曹操最活跃？", result, evidence))

,章号,曹操
0,30,42
1,40,37
2,43,35
3,67,33
4,42,30


依据： 依据 outputs/tables/character_trends.csv 中 曹操 列按章统计的出场次数（别名归一 + 长别名优先匹配）。


## 问题 3：谁与诸葛亮共现最多？

In [5]:
result, evidence = qa.cooccur_with("诸葛亮", 5)
display(result)
print("依据：", evidence)
answers.append(("谁与诸葛亮共现最多？", result, evidence))

,共现人物,共现次数
0,孙权,49
1,曹操,47
2,刘备,44
3,赵云,41
4,魏延,37


依据： 依据 outputs/tables/character_cooccurrence_edges.csv（同章共现窗口，统计与 诸葛亮 同章出现的次数）。


## 问题 4：与第 37 回（三顾茅庐）最相似的章节有哪些？

In [6]:
result, evidence = qa.similar_chapters(37, 5)
display(result)
print("依据：", evidence)
answers.append(("与第 37 回最相似的章节有哪些？", result, evidence))

,相似章节,相似度
0,36,0.422260
1,38,0.379214
2,21,0.344489
3,39,0.307160
4,42,0.266358


依据： 依据 outputs/tables/chapter_similarity.csv（TF-IDF 余弦相似度，第 37 回与其它各回两两比较）。


## 问题 5：哪些章节最像战争篇？

In [7]:
result, evidence = qa.war_chapters(5)
display(result)
print("依据：", evidence)
answers.append(("哪些章节最像战争篇？", result, evidence))

,章号,战争,战争_占比
0,40,388,0.036970
1,99,357,0.063853
2,102,354,0.054095
3,95,350,0.058804
4,101,342,0.060424


依据： 依据 outputs/tables/topic_trends.csv，按“战争”主题关键词计数排序（词典见 data/lexicons/topic_lexicon.json）。


In [8]:
from sanguo_nlp.paths import OUTPUTS_DIR

report = qa.report(answers)
(OUTPUTS_DIR / "qa_report.md").write_text(report, encoding="utf-8")
print("已保存 outputs/qa_report.md")
print(report[:800])

已保存 outputs/qa_report.md
# 《三国演义》可追溯问答报告

## 谁出场最多？

| 人物 | 频次 |
| --- | --- |
| 刘备 | 2224 |
| 诸葛亮 | 1901 |
| 曹操 | 1037 |
| 关羽 | 981 |
| 赵云 | 447 |

**依据：** 依据 outputs/tables/character_frequencies.csv（别名归一后全书提及次数）；可用 outputs/tables/character_trends.csv 的逐章求和交叉验证：刘备=2211、诸葛亮=1880、曹操=1013、关羽=969、赵云=441

## 哪几章曹操最活跃？

| 章号 | 曹操 |
| --- | --- |
| 30 | 42 |
| 40 | 37 |
| 43 | 35 |
| 67 | 33 |
| 42 | 30 |

**依据：** 依据 outputs/tables/character_trends.csv 中 曹操 列按章统计的出场次数（别名归一 + 长别名优先匹配）。

## 谁与诸葛亮共现最多？

| 共现人物 | 共现次数 |
| --- | --- |
| 孙权 | 49 |
| 曹操 | 47 |
| 刘备 | 44 |
| 赵云 | 41 |
| 魏延 | 37 |

**依据：** 依据 outputs/tables/character_cooccurrence_edges.csv（同章共现窗口，统计与 诸葛亮 同章出现的次数）。

## 与第 37 回最相似的章节有哪些？

| 相似章节 | 相似度 |
| --- | --- |
| 36 | 0.4222600543025617 |
| 38 | 0.3792142713729871 |
| 21 | 0.3444891358132904 |
| 39 | 0.30716


## 总结与实验记录

- **数据驱动的回答**：所有统计值都来自前面实验保存的 CSV，助手本身不产生任何新数字。
- **AI 帮了什么**：生成问答函数的框架、帮助润色结论、提示可以交叉验证的字段。
- **必须由自己判断的部分**：问题的定义（“出场最多”用什么口径）、证据是否充分、结论与原文阅读感受是否一致；如果答案与直觉不符，优先检查数据清洗、别名归一规则，最后才是解释。
- 当系统答案与人工阅读感受不一致时，排查顺序建议是：**数据 → 规则 → 解释**，而不是直接相信或否定模型输出。